In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
from utils import *
from scipy.stats import ttest_rel

X, y, exposure = load_data()
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exposure, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

feature_sets = {
    "traditional": TRADITIONAL_COLS,
    "telematics":  [c for c in X.columns if c not in TRADITIONAL_COLS],
    "combined":    list(X.columns),
}

# compare by CV, training data only
results = {}
for name, cols in feature_sets.items():
    alpha, folds = evaluate(X_train[cols], y_train, exp_train)
    results[name] = {"alpha": alpha, "folds": folds}
    print(f"{name}: alpha={alpha:.4g}, mean deviance={folds.mean():.4f}")

# paired check against combined
baseline = "combined"
base = results[baseline]["folds"]
rows = []
for name, r in results.items():
    if name == baseline:
        continue
    gap = r["folds"] - base
    t, p = ttest_rel(r["folds"], base)
    rows.append({
        "model": name,
        "mean_dev": r["folds"].mean(),
        "gap_vs_combined": gap.mean(),
        "gap_pct": 100 * gap.mean() / base.mean(),
        "gap_min": gap.min(),
        "gap_max": gap.max(),
        "folds_worse_than_baseline": f"{(gap > 0).sum()}/{len(gap)}",
        "t": t,
        "p": p,
    })
pd.DataFrame(rows).round(4)



traditional: alpha=0.003162, mean deviance=0.3125
telematics: alpha=0.001, mean deviance=0.2855
combined: alpha=0.001, mean deviance=0.2798


,model,mean_dev,gap_vs_combined,gap_pct,gap_min,gap_max,folds_worse_than_baseline,t,p
0,traditional,0.3125,0.0326,11.6589,0.0294,0.0370,5/5,20.9508,0.0000
1,telematics,0.2855,0.0057,2.0299,0.0036,0.0069,5/5,9.5733,0.0007


In [6]:
# test set
cols = feature_sets["combined"]
cat_cols = X[cols].select_dtypes(include=["object", "category"]).columns.tolist()
pipe = build_pipeline(cat_cols, results["combined"]["alpha"])
pipe.fit(X_train[cols], y_train / exp_train, sample_weight=exp_train)
print("Test deviance:", mean_poisson_deviance(y_test / exp_test, pipe.predict(X_test[cols]), sample_weight=exp_test))

coefs = pd.Series(pipe[-1].coef_, index=pipe[:-1].get_feature_names_out())
print(coefs.sort_values(key=abs, ascending=False))

Test deviance: 0.2810441913852901
Annual.pct.driven     0.581366
Car.age              -0.259757
Credit.score         -0.234109
Total.miles.driven    0.195781
Pct.drive.thr         0.178147
                        ...   
Territory_63          0.000592
Territory_82         -0.000327
Territory_39         -0.000246
Territory_84         -0.000241
Territory_78          0.000238
Length: 81, dtype: float64
